[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/02_Serialization/Serialization_Apply.ipynb)

# 1.2 ONNX Serialization — Hands-On Practice

Practice saving, loading, and inspecting ONNX models and tensors. Build intuition for the protobuf serialization pipeline.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup](#section-1) | Imports and helpers |
| 2 | [Exercise 1: Save & Load Round-Trip](#section-2) | Full serialize → file → deserialize cycle |
| 3 | [Exercise 2: Tensor Serialization](#section-3) | Save/load NumPy arrays as TensorProto |
| 4 | [Exercise 3: Compare File Sizes](#section-4) | How weight dimensions affect .onnx size |
| 5 | [Exercise 4: Serialize Individual Components](#section-5) | Nodes, graphs, and tensors independently |
| 6 | [Exercise 5: Verify Loaded Model Inference](#section-6) | End-to-end validation |
| 7 | [Exercise 6: File Size vs Data Type](#section-7) | float32 vs float16 vs int8 |
| 8 | [Visualization: Serialization Pipeline](#section-8) | Interactive pipeline diagram |
| 9 | [Challenge: Model Registry](#section-9) | Build a simple model versioning system |

<a id='section-1'></a>
## Section 1: Setup

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import os
import time
import glob

from onnx import TensorProto, load, save
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid, set_model_props)
from onnx.checker import check_model
from onnx import numpy_helper
from onnx.numpy_helper import from_array, to_array
import onnxruntime as ort

def build_lr_model():
    """Build a simple Y = XA + B model for testing."""
    X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
    A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
    B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
    Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])
    g = make_graph(
        [make_node('MatMul', ['X', 'A'], ['XA']),
         make_node('Add', ['XA', 'B'], ['Y'])],
        'lr', [X, A, B], [Y])
    m = make_model(g)
    check_model(m)
    return m

print('Setup complete!')

<a id='section-2'></a>
## Section 2: Exercise 1 — Save & Load Round-Trip

### Task

Build a model, save it to disk using **both** methods (low-level protobuf and high-level `onnx.save`), load both copies, and verify they are byte-identical.

In [ ]:
model = build_lr_model()

# Method 1: Low-level
with open('method1.onnx', 'wb') as f:
    f.write(model.SerializeToString())

# Method 2: High-level
save(model, 'method2.onnx')

# Load both back
with open('method1.onnx', 'rb') as f:
    loaded1 = load(f)
loaded2 = load('method2.onnx')

# Compare
bytes1 = loaded1.SerializeToString()
bytes2 = loaded2.SerializeToString()

print(f'Method 1 file size: {os.path.getsize("method1.onnx")} bytes')
print(f'Method 2 file size: {os.path.getsize("method2.onnx")} bytes')
print(f'Byte-identical:     {bytes1 == bytes2}')
print(f'Graph nodes:        {[n.op_type for n in loaded1.graph.node]}')

<a id='section-3'></a>
## Section 3: Exercise 2 — Tensor Serialization

### Task

Create a weight matrix, save it as a standalone `.pb` file, load it back, and verify the round-trip is lossless.

In [ ]:
# Create test data with various dtypes
test_tensors = {
    'weights_f32': np.random.randn(10, 5).astype(np.float32),
    'biases_f32':  np.random.randn(5).astype(np.float32),
    'indices_i64': np.array([0, 3, 7, 15, 31], dtype=np.int64),
    'mask_bool':   np.array([True, False, True, True, False]),
}

print(f'{"Name":>15s} | {"Shape":>12s} | {"DType":>8s} | {"File Size":>10s} | Roundtrip')
print('-' * 70)

for name, arr in test_tensors.items():
    proto = from_array(arr, name=name)
    fname = f'{name}.pb'

    with open(fname, 'wb') as f:
        f.write(proto.SerializeToString())

    loaded = TensorProto()
    with open(fname, 'rb') as f:
        loaded.ParseFromString(f.read())
    restored = to_array(loaded)

    ok = np.array_equal(arr, restored)
    fsize = os.path.getsize(fname)
    print(f'{name:>15s} | {str(arr.shape):>12s} | {str(arr.dtype):>8s} | '
          f'{fsize:>7d}  B | {ok}')

<a id='section-4'></a>
## Section 4: Exercise 3 — Compare File Sizes

### Task

Build models with different weight matrix sizes and visualize how file size scales with parameter count.

In [ ]:
configs = [(10, 5), (50, 25), (100, 50), (200, 100), (500, 250), (1000, 500)]
file_sizes = []
param_counts = []

for rows, cols in configs:
    W = from_array(np.random.randn(rows, cols).astype(np.float32), name='W')
    b = from_array(np.random.randn(cols).astype(np.float32), name='b')

    _X = make_tensor_value_info('X', TensorProto.FLOAT, [None, rows])
    _Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, cols])

    _g = make_graph(
        [make_node('MatMul', ['X', 'W'], ['XW']),
         make_node('Add', ['XW', 'b'], ['Y'])],
        'g', [_X], [_Y], [W, b])
    _m = make_model(_g)

    fname = f'model_{rows}x{cols}.onnx'
    save(_m, fname)
    size = os.path.getsize(fname)
    params = rows * cols + cols

    file_sizes.append(size)
    param_counts.append(params)
    print(f'  W[{rows:>4d}×{cols:>4d}]  params={params:>8,}  '
          f'file={size:>10,} bytes ({size/1024:.1f} KB)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(param_counts, [s/1024 for s in file_sizes], 'o-',
        color='#2E86C1', linewidth=2, markersize=8)
ax.plot(param_counts, [p*4/1024 for p in param_counts], 's--',
        color='#E74C3C', linewidth=2, markersize=6, label='Theoretical min (params × 4B)')
ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('File Size (KB)', fontsize=12)
ax.set_title('ONNX File Size vs Parameter Count', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='section-5'></a>
## Section 5: Exercise 4 — Serialize Individual Components

### Task

Extract a single `NodeProto` from a model, serialize it independently, and reconstruct it from bytes.

In [ ]:
model = build_lr_model()

# Extract and serialize the MatMul node
matmul_node = model.graph.node[0]
node_bytes = matmul_node.SerializeToString()

print(f'Original MatMul node:')
print(f'  op_type: {matmul_node.op_type}')
print(f'  inputs:  {list(matmul_node.input)}')
print(f'  outputs: {list(matmul_node.output)}')
print(f'  serialized size: {len(node_bytes)} bytes')

# Reconstruct from bytes
from onnx import NodeProto
restored_node = NodeProto()
restored_node.ParseFromString(node_bytes)

print(f'\nRestored node:')
print(f'  op_type: {restored_node.op_type}')
print(f'  inputs:  {list(restored_node.input)}')
print(f'  outputs: {list(restored_node.output)}')
print(f'  match:   {matmul_node.SerializeToString() == restored_node.SerializeToString()}')

<a id='section-6'></a>
## Section 6: Exercise 5 — Verify Loaded Model Inference

### Task

Build a model with embedded weights, save it, load it in a separate session, and verify the loaded model produces identical results.

In [ ]:
# Build model with embedded weights
W_data = np.array([[0.5, -0.3], [0.2, 0.8], [-0.1, 0.4]], dtype=np.float32)
b_data = np.array([0.1, -0.2], dtype=np.float32)

W_init = from_array(W_data, name='W')
b_init = from_array(b_data, name='b')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 3])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph = make_graph(
    [make_node('MatMul', ['X', 'W'], ['XW']),
     make_node('Add', ['XW', 'b'], ['Y'])],
    'weighted_lr', [X], [Y], [W_init, b_init])
model = make_model(graph)
check_model(model)

# Save and reload
save(model, 'weighted_model.onnx')
loaded = load('weighted_model.onnx')

# Create sessions from BOTH
sess_orig = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])
sess_loaded = ort.InferenceSession(
    loaded.SerializeToString(), providers=['CPUExecutionProvider'])

# Run inference
x_test = np.random.randn(5, 3).astype(np.float32)
res_orig = sess_orig.run(None, {'X': x_test})[0]
res_loaded = sess_loaded.run(None, {'X': x_test})[0]

print('Input shape:', x_test.shape)
print('Original output (first 3):', res_orig[:3])
print('Loaded output (first 3):  ', res_loaded[:3])
print('Max abs difference:', np.abs(res_orig - res_loaded).max())
print('Identical results:', np.allclose(res_orig, res_loaded))

<a id='section-7'></a>
## Section 7: Exercise 6 — File Size vs Data Type

### Task

Compare how different data types affect the serialized file size for the same logical model.

In [ ]:
n_params = 10000  # 100×100 weight matrix

dtype_configs = [
    ('float64', np.float64, TensorProto.DOUBLE),
    ('float32', np.float32, TensorProto.FLOAT),
    ('float16', np.float16, TensorProto.FLOAT16),
    ('int32',   np.int32,   TensorProto.INT32),
    ('int8',    np.int8,    TensorProto.INT8),
]

sizes = []
names = []

for dtype_name, np_dtype, onnx_dtype in dtype_configs:
    weights = (np.random.randn(100, 100) * 10).astype(np_dtype)
    W = from_array(weights, name='W')

    _X = make_tensor_value_info('X', onnx_dtype, [None, 100])
    _Y = make_tensor_value_info('Y', onnx_dtype, [None, 100])
    _g = make_graph(
        [make_node('MatMul', ['X', 'W'], ['Y'])],
        'g', [_X], [_Y], [W])
    _m = make_model(_g)

    size = len(_m.SerializeToString())
    sizes.append(size)
    names.append(dtype_name)
    print(f'  {dtype_name:>8s}: {size:>8,} bytes  '
          f'({size/1024:.1f} KB, {weights.itemsize} bytes/element)')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, [s/1024 for s in sizes],
              color=['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6'])
ax.set_xlabel('Data Type', fontsize=12)
ax.set_ylabel('File Size (KB)', fontsize=12)
ax.set_title(f'Serialized Model Size by Data Type (100×100 = {n_params:,} params)',
             fontsize=13, fontweight='bold')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{size/1024:.1f} KB', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Visualization — Serialization Pipeline

Let's create a visual showing the complete serialization pipeline with actual byte counts.

In [ ]:
model = build_lr_model()
model_bytes = model.SerializeToString()

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 5)

box = dict(boxstyle='round,pad=0.6', lw=2)
arrow_kw = dict(arrowstyle='->', color='#2C3E50', lw=2.5, mutation_scale=20)

# Python objects
ax.text(1.5, 3, 'Python\nModelProto\n(in memory)',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox={**box, 'fc': '#AED6F1', 'ec': '#2471A3'})

# Bytes
ax.text(5.5, 3, f'bytes\n{len(model_bytes)} B\n(in memory)',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox={**box, 'fc': '#F9E79F', 'ec': '#B7950B'})

# File
ax.text(9.5, 3, f'.onnx file\n{len(model_bytes)} B\n(on disk)',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox={**box, 'fc': '#A9DFBF', 'ec': '#1E8449'})

# ORT session
ax.text(13, 3, 'ORT\nSession\n(ready)',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox={**box, 'fc': '#D7BDE2', 'ec': '#7D3C98'})

# Forward arrows
ax.annotate('', xy=(3.8, 3.3), xytext=(3.0, 3.3), arrowprops=arrow_kw)
ax.text(3.4, 3.9, 'SerializeToString()', fontsize=8, ha='center', style='italic')

ax.annotate('', xy=(7.8, 3.3), xytext=(7.0, 3.3), arrowprops=arrow_kw)
ax.text(7.4, 3.9, 'write()', fontsize=8, ha='center', style='italic')

ax.annotate('', xy=(11.5, 3.3), xytext=(10.8, 3.3), arrowprops=arrow_kw)
ax.text(11.1, 3.9, 'ort.InferenceSession()', fontsize=8, ha='center', style='italic')

# Backward arrows
ax.annotate('', xy=(3.0, 2.7), xytext=(3.8, 2.7),
           arrowprops={**arrow_kw, 'color': '#E74C3C'})
ax.text(3.4, 2.1, 'ParseFromString()', fontsize=8, ha='center',
        style='italic', color='#E74C3C')

ax.annotate('', xy=(7.0, 2.7), xytext=(7.8, 2.7),
           arrowprops={**arrow_kw, 'color': '#E74C3C'})
ax.text(7.4, 2.1, 'read()', fontsize=8, ha='center',
        style='italic', color='#E74C3C')

ax.set_title('ONNX Serialization Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Challenge — Model Registry

### Task

Build a simple model registry that saves multiple versions of a model with metadata, and can list/load any version.

In [ ]:
import json
from datetime import datetime

REGISTRY_DIR = 'model_registry'
os.makedirs(REGISTRY_DIR, exist_ok=True)

def register_model(model, name, version, description=''):
    """Save a model with metadata to the registry."""
    fname = f'{name}_v{version}.onnx'
    path = os.path.join(REGISTRY_DIR, fname)
    save(model, path)

    meta = {
        'name': name,
        'version': version,
        'description': description,
        'file': fname,
        'size_bytes': os.path.getsize(path),
        'n_nodes': len(model.graph.node),
        'n_inputs': len(model.graph.input),
        'timestamp': datetime.now().isoformat(),
    }

    meta_path = os.path.join(REGISTRY_DIR, f'{name}_v{version}.json')
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    return meta

def load_registered_model(name, version):
    """Load a model from the registry."""
    path = os.path.join(REGISTRY_DIR, f'{name}_v{version}.onnx')
    return load(path)

def list_registry():
    """List all registered models."""
    entries = []
    for f in sorted(glob.glob(os.path.join(REGISTRY_DIR, '*.json'))):
        with open(f) as fh:
            entries.append(json.load(fh))
    return entries

# Register 3 versions with different architectures
m1 = build_lr_model()
meta1 = register_model(m1, 'linear_reg', 1, 'Basic Y=XA+B')

# v2: model with Abs
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])
g2 = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['XAB']),
     make_node('Abs', ['XAB'], ['Y'])],
    'lr_abs', [X, A, B], [Y])
m2 = make_model(g2)
meta2 = register_model(m2, 'linear_reg', 2, 'Y=|XA+B| with Abs')

# v3: model with Relu
g3 = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['XAB']),
     make_node('Relu', ['XAB'], ['Y'])],
    'lr_relu', [X, A, B], [Y])
m3 = make_model(g3)
meta3 = register_model(m3, 'linear_reg', 3, 'Y=Relu(XA+B)')

print('Registered Models:')
print('-' * 70)
for entry in list_registry():
    print(f"  v{entry['version']}: {entry['description']:25s} "
          f"({entry['n_nodes']} nodes, {entry['size_bytes']} bytes)")

# Load and verify v2
loaded_v2 = load_registered_model('linear_reg', 2)
print(f'\nLoaded v2: {[n.op_type for n in loaded_v2.graph.node]}')

In [ ]:
# Cleanup
import shutil
for f in glob.glob('*.onnx') + glob.glob('*.pb'):
    os.remove(f)
if os.path.exists(REGISTRY_DIR):
    shutil.rmtree(REGISTRY_DIR)
print('Cleanup complete!')

---

## Summary

| Exercise | Skill Practiced |
|----------|----------------|
| 1 | Save/load round-trip with both methods |
| 2 | Tensor serialization with various dtypes |
| 3 | File size analysis with visualization |
| 4 | Component-level serialization |
| 5 | End-to-end inference verification |
| 6 | Data type impact on file size |
| 7 | Serialization pipeline visualization |
| Challenge | Model versioning registry |

**Next:** [Initializers and Attributes](../03_Initializers_and_Attributes/) — Embed weights and set operator parameters.